In [1]:
import copy
import os
import time
from functools import partial
from pathlib import Path

import numpy as np
import torch
import torch.ao.quantization
from transformers import AutoModel, AutoTokenizer

In [2]:
device = "cpu"
device

'cpu'

In [3]:
MODEL_NAME = "sentence-transformers/multi-qa-mpnet-base-cos-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)

In [4]:
sentence_a = "I go to school every day because I want to learn new things."
sentence_b = "My brother likes playing computer games in his free time."
sentences = [sentence_a, sentence_b]

encodings = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to(
    device
)

tokens_a = tokenizer.convert_ids_to_tokens(encodings["input_ids"][0])
print(tokens_a)

tokens_b = tokenizer.convert_ids_to_tokens(encodings["input_ids"][1])
print(tokens_b)

['<s>', 'i', 'go', 'to', 'school', 'every', 'day', 'because', 'i', 'want', 'to', 'learn', 'new', 'things', '.', '</s>']
['<s>', 'my', 'brother', 'likes', 'playing', 'computer', 'games', 'in', 'his', 'free', 'time', '.', '</s>', '<pad>', '<pad>', '<pad>']


## 1. Evaluation mode

In [5]:
def benchmark(
    fn,
    device,
    fn_setup=lambda: None,
    fn_cleanup=lambda: None,
    n_runs=100,
    n_warmup=10,
):
    fn_setup()

    for _ in range(n_warmup):
        fn()

    if device == "cuda":
        torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)

    fn_cleanup()
    return np.mean(times), np.std(times)


def benchmark_model(fn, model, inputs, **kwargs):
    """Benchmark `fn(model, inputs)` on the device the model currently lives on (cpu / cuda)."""
    model_device = next(model.parameters()).device.type
    assert all(v.device.type == model_device for v in inputs.values()), (
        "model and inputs are on different devices"
    )

    return benchmark(partial(fn, model, inputs), device=model_device, **kwargs)


def print_benchmark_results(results, baseline):
    baseline_mean = results[baseline][0]

    width = max(len(name) for name in results) + 2

    print(f"{'mode':<{width}}{'mean [ms]':>10}{'std [ms]':>10}{'speedup':>9}")
    for name, (mean, std) in results.items():
        print(f"{name:<{width}}{mean:>10.2f}{std:>10.2f}{baseline_mean / mean:>8.2f}x")

In [6]:
mode = "training" if model.training else "eval"
print(f"Model is currently in {mode} mode.")

Model is currently in eval mode.


In [7]:
# from_pretrained() returns the model already in eval mode

def no_optimization(model, inputs):
    model(**inputs)


def eval_mode(model, inputs):
    model(**inputs)


def eval_mode_and_no_grad(model, inputs):
    with torch.no_grad():
        model(**inputs)


def eval_mode_and_inference_mode(model, inputs):
    with torch.inference_mode():
        model(**inputs)

In [8]:
results = {}

results["no optimization"] = benchmark_model(
    no_optimization, model, encodings, fn_setup=model.train, fn_cleanup=model.eval
)
results["no optimization"]

(np.float64(61.00492475012288), np.float64(38.81563216936445))

In [9]:
results["eval()"] = benchmark_model(
    eval_mode, model, encodings, fn_setup=model.eval
)
results["eval()"]

(np.float64(44.972006659972976), np.float64(12.05478333432018))

In [10]:
results["eval() + no_grad()"] = benchmark_model(
    eval_mode_and_no_grad, model, encodings, fn_setup=model.eval
)
results["eval() + no_grad()"]

(np.float64(70.94467485001587), np.float64(66.39097152527262))

In [11]:
results["eval() + inference_mode()"] = benchmark_model(
    eval_mode_and_inference_mode, model, encodings, fn_setup=model.eval
)
results["eval() + inference_mode()"]

(np.float64(54.234690250068525), np.float64(40.19170815081203))

In [12]:
print_benchmark_results(results, baseline="no optimization")

mode                        mean [ms]  std [ms]  speedup
no optimization                 61.00     38.82    1.00x
eval()                          44.97     12.05    1.36x
eval() + no_grad()              70.94     66.39    0.86x
eval() + inference_mode()       54.23     40.19    1.12x


## 2. PyTorch model compilation

In [13]:
model.eval()
model_compiled = torch.compile(model)

In [14]:
start = time.perf_counter()
eval_mode_and_inference_mode(model_compiled, encodings)
compile_time_s = time.perf_counter() - start

print(f"Compilation + warm-up: {compile_time_s:.1f} s")

Compilation + warm-up: 11.8 s


In [15]:
results["compiled + eval() + inference_mode()"] = benchmark_model(
    eval_mode_and_inference_mode, model_compiled, encodings
)
results["compiled + eval() + inference_mode()"]

(np.float64(40.66001468007016), np.float64(21.59982983927104))

In [16]:
print_benchmark_results(results, baseline="no optimization")

mode                                   mean [ms]  std [ms]  speedup
no optimization                            61.00     38.82    1.00x
eval()                                     44.97     12.05    1.36x
eval() + no_grad()                         70.94     66.39    0.86x
eval() + inference_mode()                  54.23     40.19    1.12x
compiled + eval() + inference_mode()       40.66     21.60    1.50x


## 3. Quantization

In [17]:
# Dynamic quantization works on CPU - the model is already there (device = "cpu").
model = model.to("cpu")

model_quantized = torch.ao.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
print(model_quantized)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (k): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (v): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (o): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (dropout): Dropout(p=0.1, inplace=F

In [18]:
MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(exist_ok=True)


def save_and_get_size_mb(model, path):
    torch.save(model.state_dict(), path)
    return os.path.getsize(path) / (1024**2)

In [19]:
sizes = {}

sizes["Original model"] = save_and_get_size_mb(model, MODEL_DIR / "model_fp32.pt")
sizes["Quantized model"] = save_and_get_size_mb(
    model_quantized, MODEL_DIR / "model_quantized.pt"
)

baseline_size = sizes["Original model"]

In [20]:
print(f"{'model':<38}{'size [MB]':>10}{'compression':>13}")
for name, size in sizes.items():
    print(f"{name:<38}{size:>10.2f}{baseline_size / size:>12.2f}x")

model                                  size [MB]  compression
Original model                            417.73        1.00x
Quantized model                           173.10        2.41x


In [21]:
# os.remove(MODEL_DIR / "model_fp32.pt")
# os.remove(MODEL_DIR / "model_quantized.pt")

In [22]:
results_cpu = {}

results_cpu["Original model"] = benchmark_model(
    eval_mode_and_inference_mode, model, encodings
)
results_cpu["Quantized model"] = benchmark_model(
    eval_mode_and_inference_mode, model_quantized, encodings
)

In [23]:
print_benchmark_results(results_cpu, baseline="Original model")

mode              mean [ms]  std [ms]  speedup
Original model        50.64     20.38    1.00x
Quantized model       22.90      6.36    2.21x


## 4. GPU optimization strategies

In [24]:
torch._dynamo.reset()

device = "cuda"
model = model.to(device).eval()
encodings = encodings.to(device)

torch.cuda.empty_cache()
print("Model on", next(model.parameters()).device)

Model on cuda:0


In [ ]:
texts = {
    "short": sentence_a,
    "medium": " ".join([sentence_a] * 4),
    "long": " ".join([sentence_a] * 16),
}

inputs_by_size = {}
for size_name, text in texts.items():
    inputs_by_size[size_name] = tokenizer(
        [text, text], return_tensors="pt", padding=True, truncation=True
    ).to(device)

# Our data is already on GPU, so there is no need to use `pin_memory`.

In [26]:
compile_variants = {
    "eager (no compile)": None,
    "compile (default)": {},
    'compile "max-autotune"': {"mode": "max-autotune"},
    'compile "max-autotune-no-cudagraphs"': {"mode": "max-autotune-no-cudagraphs"},
}

results_gpu = {size_name: {} for size_name in inputs_by_size}

for variant, compile_kwargs in compile_variants.items():
    torch._dynamo.reset()
    model_variant = model if compile_kwargs is None else torch.compile(model, **compile_kwargs)

    for size_name, inputs in inputs_by_size.items():
        results_gpu[size_name][variant] = benchmark_model(
            eval_mode_and_inference_mode, model_variant, inputs
        )

W1121 14:37:49.804000 23449 torch/_inductor/utils.py:1048] [0/0] Not enough SMs to use max_autotune_gemm mode


In [27]:
for size_name, res in results_gpu.items():
    print(f"\n=== Input: {size_name} ===")
    print_benchmark_results(res, baseline="compile (default)")


=== Input: short ===
mode                                   mean [ms]  std [ms]  speedup
eager (no compile)                          9.85      2.95    0.59x
compile (default)                           5.86      1.36    1.00x
compile "max-autotune"                      4.00      0.24    1.46x
compile "max-autotune-no-cudagraphs"        5.47      0.81    1.07x

=== Input: medium ===
mode                                   mean [ms]  std [ms]  speedup
eager (no compile)                         13.61      0.98    0.72x
compile (default)                           9.76      2.08    1.00x
compile "max-autotune"                      9.54      0.41    1.02x
compile "max-autotune-no-cudagraphs"       10.09      0.50    0.97x

=== Input: long ===
mode                                   mean [ms]  std [ms]  speedup
eager (no compile)                         37.59      1.18    0.90x
compile (default)                          33.90      0.62    1.00x
compile "max-autotune"                     33.99  

`max-autotune` sped up only the short input (1.46x, and about 1.0x for medium and long), while `max-autotune-no-cudagraphs` was practically equal to default.

## 4. Changing numerical precision